# VACV SI v23 — честный прогноз SI из SMILES (Kaggle)

**Задача:** прогнозировать `SI VACV` по молекулярной структуре без использования запрещённых `CC50_uM`, `IC50_uM_VACV` ни при обучении, ни при предсказании.

SI — отношение экспериментальных концентраций, **точно вывести его из одной структуры нельзя**: мы учим оценку SI на реальных обучающих SI и дополняем подтверждёнными внешними SI из ChEMBL. Разрешённые поля конкурсных файлов: `SMILES`, `SI VACV` только для train; `index`, `SMILES` для test.

**Что добавлено к v21:** отдельный источник-зависимый Gaussian Process с Morgan Tanimoto (radius 2, bit, alpha=0.1); сравнение со старым GP через grouped scaffold/Butina CV; основной осторожный ансамбль + отдельные контрольные/агрессивные предсказания. Новое направление «zero/positive hurdle» и KNN пробовались локально и уступили GP по MSE, поэтому в конкурсную ветку не включены.

**Важное ограничение:** это development cross-validation; её показатели нельзя напрямую приравнивать к скрытому тесту Kaggle. SCALE старого v21 сохранён для честного контрольного воспроизведения, а не вычислен из запрещённых концентраций.

Kaggle: прикрепите `train_v2.csv`, `test.csv`, `sample_submit.csv`, CPU, `Run All`. Встроенный снимок ChEMBL воспроизводит прежний аудит без интернета. При выборе `SOURCE_MODE='download'` нужен Internet On.


In [ ]:
import sys,subprocess,importlib.util,os
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
os.environ.setdefault('OMP_NUM_THREADS','2')
packages={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn',
          'rdkit':'rdkit','requests':'requests','threadpoolctl':'threadpoolctl'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])


In [ ]:
from pathlib import Path
import json,time,hashlib,platform,importlib.metadata as meta
import numpy as np,pandas as pd
from IPython.display import display,FileLink
WORK=Path('/kaggle/working/vacv_v23/v21') if Path('/kaggle').exists() else Path.cwd()/'vacv_v23'/'v21'
WORK.mkdir(parents=True,exist_ok=True);sys.path.insert(0,str(WORK))
INPUT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
TRAIN=TEST=SAMPLE=None
# For local reproductions with attached input files, set exact paths when present.
# Kaggle inputs are detected separately below.
if not Path('/kaggle/input').exists() and Path('/mnt/data/train_v2(2).csv').exists():
    TRAIN='/mnt/data/train_v2(2).csv'
    TEST='/mnt/data/test(2).csv'
    SAMPLE='/mnt/data/sample_submit(2).csv' 
SOURCE_MODE='snapshot' # 'download': запросить актуальные реальные записи ChEMBL.
OUTER_FOLDS=5
INNER_FOLDS=3


In [ ]:
(WORK/'vacv_core.py').write_text('# Structural primitives extracted unchanged from user v13.1. Legacy validation and external ingestion removed.\nfrom __future__ import annotations\n\nimport argparse, json, platform, time\n\nfrom pathlib import Path\n\nimport numpy as np, pandas as pd\n\nfrom scipy.stats import spearmanr\n\nfrom rdkit import Chem, DataStructs, RDLogger, rdBase\n\nfrom rdkit.Chem import MACCSkeys, rdFingerprintGenerator, rdMolDescriptors\n\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\n\nfrom rdkit.ML.Cluster import Butina\n\nRDLogger.DisableLog(\'rdApp.*\')\n\nSMI, TGT = \'SMILES\', \'SI VACV\'\n\ndef canon(s):\n    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None\n    return Chem.MolToSmiles(m, isomericSmiles=True) if m is not None else None\n\ndef scaffold(s):\n    return MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(s)) or \'ACYCLIC\'\n\ndef load(train_csv, test_csv, sample_csv):\n    tr = pd.read_csv(train_csv, usecols=[SMI, TGT]); te = pd.read_csv(test_csv, usecols=[\'index\', SMI])\n    sm = pd.read_csv(sample_csv, usecols=[\'index\'])           # IDs/order only - never the sample\'s values\n    tr[TGT] = pd.to_numeric(tr[TGT], errors=\'coerce\'); tr[\'c\'] = tr[SMI].map(canon); te[\'c\'] = te[SMI].map(canon)\n    bad = tr.c.isna() | ~np.isfinite(tr[TGT]) | (tr[TGT] < 0)\n    if bad.any(): print(\'dropping invalid train rows:\', int(bad.sum())); tr = tr[~bad].reset_index(drop=True)\n    if te.c.isna().any(): raise ValueError(\'invalid test SMILES\')\n    if not (te[\'index\'].is_unique and sm[\'index\'].is_unique and set(te[\'index\']) == set(sm[\'index\'])): raise ValueError(\'test/sample index mismatch\')\n    tr[\'scaf\'] = tr.c.map(scaffold)\n    return tr, te, sm\n\ndef _bulk(fps): return np.array([DataStructs.BulkTanimotoSimilarity(f, fps) for f in fps])\n\ndef build_kernels(smiles):\n    mols = [Chem.MolFromSmiles(s) for s in smiles]\n    K = {}\n    for r in (2, 3):\n        g = rdFingerprintGenerator.GetMorganGenerator(radius=r, fpSize=2048, includeChirality=True)\n        K[f\'mc{r}\'] = _bulk([g.GetCountFingerprint(m) for m in mols])          # count (MinMax) Tanimoto\n    g = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)\n    K[\'mb2\'] = _bulk([g.GetFingerprint(m) for m in mols])                      # bit Tanimoto\n    K[\'maccs\'] = _bulk([MACCSkeys.GenMACCSKeys(m) for m in mols])\n    K[\'ap\'] = _bulk([rdMolDescriptors.GetHashedAtomPairFingerprint(m, 2048) for m in mols])\n    K[\'mix\'] = (K[\'mc2\'] + K[\'maccs\'] + K[\'ap\']) / 3\n    return K\n\nTF = {\'raw\': (lambda y: y, lambda z: np.maximum(z, 0)), \'sqrt\': (np.sqrt, lambda z: np.maximum(z, 0) ** 2),\n      \'q4\': (lambda y: y ** .25, lambda z: np.maximum(z, 0) ** 4), \'log\': (np.log1p, lambda z: np.expm1(np.clip(z, 0, 20)))}\n\ndef smear(tf, mu, v):\n    """E[g^-1(f)], f~N(mu, v): conditional MEAN on the SI scale (plain back-transform gives a median and is biased low)."""\n    m = np.maximum(mu, 0)\n    return {\'raw\': m, \'sqrt\': m**2 + v, \'q4\': m**4 + 6*m**2*v + 3*v**2, \'log\': np.expm1(np.minimum(m + v/2, 20))}[tf]\n\ndef krr(K, y, tri, evi, tf=\'log\', alpha=.3, gp=False):\n    f, finv = TF[tf]; yt = f(y[tri]); m = yt.mean(); a = yt.var() + 1e-12\n    A = K[np.ix_(tri, tri)] + alpha * np.eye(len(tri)); Ke = K[np.ix_(evi, tri)]\n    mu = m + Ke @ np.linalg.solve(A, yt - m)\n    if not gp: return finv(mu)\n    v = a * np.maximum(1 - np.einsum(\'ij,ji->i\', Ke, np.linalg.solve(A, Ke.T)), 0) + a * alpha  # latent + noise variance\n    return smear(tf, mu, v)\n\nGP_KERNELS, GP_ALPHAS = (\'mb2\', \'mc2\', \'mc3\', \'mix\'), (.1, .3)\n\ndef gp_members(K, y, tri, evi): return np.array([krr(K[k], y, tri, evi, \'log\', a, gp=True) for k in GP_KERNELS for a in GP_ALPHAS])\n\ndef gp_log_ens_raw(K, y, tri, evi): return gp_members(K, y, tri, evi).mean(0)\n\ndef butina_groups(Kmat, thr):\n    n = len(Kmat); d = []\n    for i in range(1, n): d.extend(1 - Kmat[i, :i])\n    cl = Butina.ClusterData(d, n, 1 - thr, isDistData=True, reordering=True)\n    g = np.zeros(n, int)\n    for j, c in enumerate(cl): g[list(c)] = j\n    return g\n',encoding="utf-8")

In [ ]:
(WORK/'fetch_sources.py').write_text("from pathlib import Path\nimport json,time,concurrent.futures\nimport requests\nAPI='https://www.ebi.ac.uk/chembl/api/data'\nDOCS=['CHEMBL2321819','CHEMBL4052722','CHEMBL4368876','CHEMBL4699424','CHEMBL4837211','CHEMBL5223146','CHEMBL6092128']\n\ndef pages(endpoint,key,params):\n    out=[];offset=0\n    while True:\n        for attempt in range(4):\n            try:\n                r=requests.get(f'{API}/{endpoint}.json',params={**params,'limit':500,'offset':offset},timeout=(15,60));r.raise_for_status();j=r.json();break\n            except requests.RequestException:\n                if attempt==3:raise\n                time.sleep(2**attempt)\n        b=j[key];out+=b;offset+=len(b)\n        if not j['page_meta'].get('next'):return out\n        if not b:raise RuntimeError('Empty nonterminal API page')\n\ndef download_doc(doc,out,refresh=False):\n    p=Path(out)/f'{doc}.json';p.parent.mkdir(parents=True,exist_ok=True)\n    if p.exists() and not refresh:return json.loads(p.read_text())\n    acts=pages('activity','activities',{'document_chembl_id':doc})\n    assays=pages('assay','assays',{'document_chembl_id':doc})\n    docs=pages('document','documents',{'document_chembl_id':doc})\n    if len(docs)!=1:raise ValueError('Ambiguous document')\n    raw={'doi':docs[0].get('doi'),'documents':docs,'activities':acts,'assays':assays}\n    p.write_text(json.dumps(raw));print(doc,'activities',len(acts),'assays',len(assays),flush=True)\n    return raw\n\ndef download(out,refresh=False):\n    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:\n        return list(ex.map(lambda d:download_doc(d,out,refresh),DOCS))\n\nif __name__=='__main__':download(Path(__file__).parent/'raw_sources')\n",encoding="utf-8")

In [ ]:
(WORK/'paired_data.py').write_text('"""External-data-only pairing and arithmetic audit. Never accepts competition CSVs."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nimport vacv_core as V\nfrom fetch_sources import API,DOCS\n\n# Explicit assay identities, not a join across arbitrary measurements.\n# Each SI assay explicitly defines CC50/IC50 for the corresponding VACV cell assay.\nPAIRS={\n \'CHEMBL2321819\':(\'CHEMBL2330536\',\'CHEMBL2330528\',\'CHEMBL2330527\',\'BSC-1\'),\n \'CHEMBL4052722\':(\'CHEMBL4056706\',\'CHEMBL4056694\',\'CHEMBL4056707\',\'CV-1\'),\n \'CHEMBL4368876\':(\'CHEMBL4369494\',\'CHEMBL4369495\',\'CHEMBL4369496\',\'Vero\'),\n \'CHEMBL4699424\':(\'CHEMBL4700293\',\'CHEMBL4700294\',\'CHEMBL4700295\',\'Vero\'),\n \'CHEMBL4837211\':(\'CHEMBL4838274\',\'CHEMBL4838273\',\'CHEMBL4838275\',\'Vero\'),\n \'CHEMBL5223146\':(\'CHEMBL5225279\',\'CHEMBL5225278\',\'CHEMBL5225280\',\'Vero\'),\n \'CHEMBL6092128\':(\'CHEMBL6092367\',\'CHEMBL6092368\',\'CHEMBL6092369\',\'Vero\'),\n}\nUNITS={\'M\':1e6,\'mM\':1000.,\'uM\':1.,\'µM\':1.,\'μM\':1.,\'nM\':.001,\'pM\':.000001}\n# One reviewed correction, guarded by activity ID, DOI and observed value.\n# Table 1 in the source uses comma as a thousands separator: 156,733.\nCORRECTION={\'activity_id\':24978340,\'doi\':\'10.1016/j.bmcl.2022.129080\',\n \'expected_value\':156.73,\'correct_value\':156733.,\n \'url\':\'https://pmc.ncbi.nlm.nih.gov/articles/PMC9674569/\',\n \'location\':\'Table 1, NIOCH-14, SI column\',\'raw_source_text\':\'156,733\',\n \'reason\':\'Thousands separator in the primary table; corroborated by CC50/IC50.\'}\n\ndef num(x):\n    try:return float(x)\n    except (TypeError,ValueError):return np.nan\n\ndef concentration(a):\n    return num(a.get(\'standard_value\'))*UNITS.get(a.get(\'standard_units\'),np.nan)\n\ndef same_cell(ass,expected):\n    normalize=lambda x:\'\'.join(c for c in str(x).lower() if c.isalnum())\n    observed=ass.get(\'assay_cell_type\')\n    if observed:return normalize(observed)==normalize(expected)\n    return normalize(expected) in normalize(ass.get(\'description\',\'\'))\n\ndef audit(raw_sources,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);rows=[];manifests=[]\n    for raw in raw_sources:\n        doc=raw[\'documents\'][0][\'document_chembl_id\'];doi=raw[\'doi\']\n        if doc not in PAIRS:raise ValueError(\'Source not reviewed: \'+doc)\n        ia,ca,sa,cell=PAIRS[doc];am={a[\'assay_chembl_id\']:a for a in raw[\'assays\']}\n        for aid in [ia,ca,sa]:\n            if aid not in am:raise ValueError(\'Missing reviewed assay \'+aid)\n            if not same_cell(am[aid],cell):raise ValueError(\'Cell metadata changed: \'+aid)\n        description=am[sa][\'description\'].lower()\n        if not all(w in description for w in [\'cc50\',\'ic50\',\'vaccinia\']):raise ValueError(\'SI definition changed\')\n        if \'vaccinia\' not in am[ia][\'description\'].lower():raise ValueError(\'Non-VACV inhibition assay\')\n        manifests.append(dict(document_id=doc,doi=doi,ic_assay=ia,cc_assay=ca,si_assay=sa,\n            cell=cell,strain=am[ia].get(\'assay_strain\'),\n            ic_description=am[ia][\'description\'],cc_description=am[ca][\'description\'],si_description=am[sa][\'description\']))\n        acts=raw[\'activities\']\n        for si in [a for a in acts if a[\'assay_chembl_id\']==sa]:\n            mol=si[\'molecule_chembl_id\'];reasons=[]\n            ic=[a for a in acts if a[\'assay_chembl_id\']==ia and a[\'molecule_chembl_id\']==mol and a[\'standard_type\']==\'IC50\']\n            cc=[a for a in acts if a[\'assay_chembl_id\']==ca and a[\'molecule_chembl_id\']==mol and a[\'standard_type\']==\'CC50\']\n            c=V.canon(si.get(\'canonical_smiles\'));raw_si=num(si.get(\'standard_value\'));reported=raw_si;correction=\'\'\n            if si[\'activity_id\']==CORRECTION[\'activity_id\']:\n                if doi.lower()!=CORRECTION[\'doi\'] or not np.isclose(raw_si,CORRECTION[\'expected_value\'],rtol=0,atol=.005):\n                    raise ValueError(\'Reviewed correction no longer matches source; re-review, never silently override.\')\n                reported=CORRECTION[\'correct_value\'];correction=\'primary_table_thousands_separator\'\n            row=dict(SMILES=c,IC50_uM=np.nan,CC50_uM=np.nan,SI=np.nan,\n                SI_reported_raw=raw_si,SI_reported_reviewed=reported,doi=doi,document_id=doc,\n                molecule_id=mol,cell=cell,strain=am[ia].get(\'assay_strain\'),\n                si_activity_id=si[\'activity_id\'],ic_activity_id=None,cc_activity_id=None,\n                ic_assay_id=ia,cc_assay_id=ca,si_assay_id=sa,\n                si_source_url=f\'{API}/activity/{si["activity_id"]}.json\',\n                correction=correction,definition=\'CC50/IC50\',origin=\'external_ChEMBL\',\n                label_origin=\'ratio_of_paired_external_concentrations\',\n                ic_description=am[ia][\'description\'],cc_description=am[ca][\'description\'])\n            if len(ic)!=1 or len(cc)!=1:reasons.append(\'missing_or_ambiguous_pair\')\n            else:\n                ic,cc=ic[0],cc[0];iv,cv=concentration(ic),concentration(cc)\n                row.update(IC50_uM=iv,CC50_uM=cv,ic_activity_id=ic[\'activity_id\'],cc_activity_id=cc[\'activity_id\'],\n                    ic_source_url=f\'{API}/activity/{ic["activity_id"]}.json\',\n                    cc_source_url=f\'{API}/activity/{cc["activity_id"]}.json\',\n                    IC50_relation=ic.get(\'standard_relation\'),CC50_relation=cc.get(\'standard_relation\'))\n                if not c or c!=V.canon(ic.get(\'canonical_smiles\')) or c!=V.canon(cc.get(\'canonical_smiles\')):\n                    reasons.append(\'structure_mismatch\')\n                if any(a.get(\'standard_relation\')!=\'=\' for a in [ic,cc,si]):reasons.append(\'censored_or_unknown_relation\')\n                if any(a.get(\'data_validity_comment\') for a in [ic,cc,si]):reasons.append(\'validity_flag\')\n                if not (np.isfinite(iv) and np.isfinite(cv) and iv>0 and cv>0):reasons.append(\'invalid_concentration_or_unit\')\n                else:\n                    ratio=cv/iv;row[\'SI\']=ratio\n                    row[\'reported_vs_ratio_factor\']=reported/ratio if np.isfinite(reported) else np.nan\n                    # Integer SI may have rounding up to 1 unit; otherwise allow 5% rounding.\n                    if not np.isfinite(reported) or abs(reported-ratio)>max(1.,.05*ratio):\n                        reasons.append(\'SI_arithmetic_conflict\')\n            row[\'accepted\']=not reasons;row[\'review_reasons\']=\';\'.join(reasons);rows.append(row)\n    review=pd.DataFrame(rows);review.to_csv(out/\'pair_audit.csv\',index=False)\n    accepted=review[review.accepted].copy();accepted.to_csv(out/\'external_pairs.csv\',index=False)\n    pd.DataFrame(manifests).to_csv(out/\'assay_pair_manifest.csv\',index=False)\n    (out/\'reviewed_corrections.json\').write_text(json.dumps([CORRECTION],indent=2))\n    report=dict(reviewed_rows=len(review),accepted_pairs=len(accepted),unique_structures=accepted.SMILES.nunique(),\n        sources=int(accepted.document_id.nunique()),corrected_rows=int((accepted.correction!=\'\').sum()),\n        rejected_reasons=review.loc[~review.accepted,\'review_reasons\'].value_counts().to_dict(),\n        scope=\'Published SI corroborated by unique concentration pair in explicitly mapped assays. No competition CC50/IC50 read. Structures are ChEMBL-curated; not all manually redrawn from papers.\')\n    (out/\'audit_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return accepted\n\nif __name__==\'__main__\':\n    root=Path(__file__).parent\n    audit([json.loads(p.read_text()) for p in sorted((root/\'raw_sources\').glob(\'*.json\'))],root/\'data_audit\')\n',encoding="utf-8")

In [ ]:
(WORK/'source_model.py').write_text('"""SMILES-only source-aware GP ensemble. No concentration column enters fitting."""\nfrom pathlib import Path\nimport json,time\nimport numpy as np\nimport pandas as pd\nfrom scipy.linalg import cho_factor,cho_solve\nfrom scipy.optimize import minimize\nfrom sklearn.model_selection import GroupKFold\nfrom threadpoolctl import threadpool_limits\nimport vacv_core as V\n\nSCALE=1.6114604791188905  # Reproduces the user\'s successful v14 experimental CSV.\nEXTERNAL_WEIGHT=.25\nSOURCE_BIAS=1.0\nSHRINK=.05\n\ndef folds(groups,k):\n    k=min(k,len(np.unique(groups)))\n    if k<2:raise ValueError(\'Not enough independent groups\')\n    return list(GroupKFold(k).split(np.zeros(len(groups)),groups=groups))\n\ndef external_for_fold(ext,tr,K,a,blocked,scheme):\n    mask=~ext.SMILES.isin(set(tr.iloc[a].c)|set(tr.iloc[blocked].c)).to_numpy()\n    if len(blocked) and scheme==\'scaffold\':\n        mask &= ~ext.scaf.isin(set(tr.iloc[blocked].scaf)).to_numpy()\n    if len(blocked) and scheme==\'cluster065\':\n        sim=K[\'mc2\'][np.ix_(len(tr)+np.arange(len(ext)),blocked)]\n        mask &= sim.max(axis=1)<.65\n    return np.where(mask)[0]\n\ndef posterior(K,y,ext,a,b,e,source_bias=True):\n    labels=np.r_[y[a],ext.SI.to_numpy()[e]];ids=np.r_[a,len(y)+e]\n    z=np.log1p(labels);prior=np.log1p(y[a]);mean=prior.mean();amp=prior.var()+1e-12\n    c=np.zeros((len(ids),len(ids)))\n    if len(e) and source_bias:\n        sources=ext.document_id.to_numpy()[e]\n        c[len(a):,len(a):]=SOURCE_BIAS*(sources[:,None]==sources[None,:])\n    out=[]\n    for name in V.GP_KERNELS:\n        for alpha in V.GP_ALPHAS:\n            noise=np.r_[np.full(len(a),alpha),np.full(len(e),alpha/EXTERNAL_WEIGHT)]\n            kt=K[name][np.ix_(ids,ids)]+c+np.diag(noise)\n            ke=K[name][np.ix_(b,ids)];factor=cho_factor(kt,lower=True,check_finite=False)\n            mu=mean+ke@cho_solve(factor,z-mean,check_finite=False)\n            var=amp*np.maximum(1-np.einsum(\'ij,ji->i\',ke,cho_solve(factor,ke.T,check_finite=False)),0)+amp*alpha\n            out.append(np.expm1(np.minimum(np.maximum(mu,0)+var/2,20)))\n    return SCALE*np.mean(out,axis=0)\n\ndef weights(P,y):\n    prior=np.zeros(P.shape[1]);prior[0]=1;scale=max(float(np.mean(y*y)),1.)\n    def f(w):return np.mean((P@w-y)**2)/scale+SHRINK*np.sum((w-prior)**2)\n    def jac(w):return 2*P.T@(P@w-y)/(len(y)*scale)+2*SHRINK*(w-prior)\n    r=minimize(f,prior,jac=jac,method=\'SLSQP\',bounds=[(0,1)]*len(prior),\n        constraints={\'type\':\'eq\',\'fun\':lambda w:w.sum()-1,\'jac\':lambda w:np.ones(len(w))},\n        options={\'maxiter\':500,\'ftol\':1e-11})\n    if not r.success:raise RuntimeError(r.message)\n    w=np.maximum(r.x,0);return w/w.sum()\n\ndef prepare_external(paired):\n    # Hard allowlist. Concentrations, metadata text and competition IDs cannot become features/targets.\n    allowed=paired.loc[:,[\'SMILES\',\'SI\',\'document_id\']].copy()\n    allowed.SMILES=allowed.SMILES.map(V.canon)\n    if allowed.SMILES.isna().any() or not np.isfinite(allowed.SI).all() or (allowed.SI<0).any():\n        raise ValueError(\'Invalid external training observations\')\n    ext=allowed.groupby([\'SMILES\',\'document_id\'],as_index=False).SI.mean()\n    ext[\'scaf\']=ext.SMILES.map(V.scaffold)\n    return ext\n\ndef make_predictor(tr,ext,smiles):\n    K=V.build_kernels(smiles);y=tr[V.TGT].to_numpy(float)\n    sources=sorted(ext.document_id.unique());names=[\'v14_best_reference\',\'all_sources\']+sources\n    def experts(a,b,blocked,scheme):\n        e=external_for_fold(ext,tr,K,a,blocked,scheme)\n        ref=SCALE*V.gp_log_ens_raw(K,y,a,b)\n        predictions=[ref,posterior(K,y,ext,a,b,e) if len(e) else ref.copy()]\n        for source in sources:\n            selected=e[ext.document_id.to_numpy()[e]==source]\n            predictions.append(posterior(K,y,ext,a,b,selected) if len(selected) else ref.copy())\n        P=np.column_stack(predictions)\n        if not np.isfinite(P).all() or (P<0).any():raise ValueError(\'Invalid model prediction\')\n        return P\n    return K,names,experts\n\ndef stats(y,p):\n    e=(y-p)**2;keep=y!=y.max()\n    return dict(mse=float(e.mean()),mae=float(np.abs(y-p).mean()),\n        mse_without_max_diagnostic=float(e[keep].mean()),max_label_sse_share=float(e[~keep].sum()/max(e.sum(),1e-12)))\n\ndef run(tr,te,sample,paired,out,outer_folds=5,inner_folds=3):\n    with threadpool_limits(limits=2):return _run(tr,te,sample,paired,out,outer_folds,inner_folds)\n\ndef _run(tr,te,sample,paired,out,outer_folds,inner_folds):\n    started=time.time();out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    ext=prepare_external(paired);n=len(tr);y=tr[V.TGT].to_numpy(float)\n    K,names,experts=make_predictor(tr,ext,list(tr.c)+ext.SMILES.tolist()+list(te.c))\n    groups={\'scaffold\':tr.scaf.to_numpy(),\'cluster065\':V.butina_groups(K[\'mc2\'][:n,:n],.65)}\n    def inner_fit(a,blocked,scheme):\n        P=np.zeros((len(a),len(names)))\n        for ia,ib in folds(groups[scheme][a],inner_folds):\n            P[ib]=experts(a[ia],a[ib],np.r_[blocked,a[ib]],scheme)\n        return weights(P,y[a])\n    rows=[];oofs=[];fold_rows=[];weight_rows=[];full_predictions={}\n    for scheme,g in groups.items():\n        P=np.zeros((n,len(names)));blend=np.zeros(n);fold_ids=np.zeros(n,int)\n        for fid,(a,b) in enumerate(folds(g,outer_folds)):\n            w=inner_fit(a,b,scheme);P[b]=experts(a,b,b,scheme);blend[b]=P[b]@w;fold_ids[b]=fid\n            weight_rows.append(dict(scheme=scheme,fold=fid,**dict(zip(names,w))))\n            for name,p in [(name,P[b,j]) for j,name in enumerate(names)]+[(\'nested_source_blend\',blend[b])]:\n                fold_rows.append(dict(scheme=scheme,fold=fid,n=len(b),model=name,mse=float(((p-y[b])**2).mean())))\n            print(scheme,fid,\'reference\',round(((P[b,0]-y[b])**2).mean()),\'blend\',round(((blend[b]-y[b])**2).mean()),flush=True)\n        for name,p in [(name,P[:,j]) for j,name in enumerate(names)]+[(\'nested_source_blend\',blend)]:\n            rows.append(dict(scheme=scheme,model=name,**stats(y,p)))\n        frame=pd.DataFrame(P,columns=names);frame[\'nested_source_blend\']=blend\n        frame[\'target\']=y;frame[\'row_id\']=np.arange(n);frame[\'fold\']=fold_ids;frame[\'scheme\']=scheme;oofs.append(frame)\n        full_predictions[scheme]=P\n    metric=pd.DataFrame(rows);metric.to_csv(out/\'metrics.csv\',index=False)\n    pd.concat(oofs).to_csv(out/\'oof.csv\',index=False);pd.DataFrame(fold_rows).to_csv(out/\'fold_metrics.csv\',index=False)\n    pd.DataFrame(weight_rows).to_csv(out/\'outer_weights.csv\',index=False)\n    w=inner_fit(np.arange(n),np.array([],int),\'scaffold\')\n    P=experts(np.arange(n),np.arange(n+len(ext),n+len(ext)+len(te)),np.array([],int),\'scaffold\')\n    blend=P@w\n    pivot=metric.pivot(index=\'scheme\',columns=\'model\',values=\'mse\')\n    gate=bool((pivot[\'nested_source_blend\']<pivot[\'v14_best_reference\']).all())\n    # Export all honest alternatives, with no claim that an unchanged control is new.\n    for name,pred in [(\'submission_v21_candidate.csv\',blend),(\'submission_v14_control.csv\',P[:,0]),\n                      (\'submission_all_sources.csv\',P[:,1]),(\'submission_selected.csv\',blend if gate else P[:,0])]:\n        f=sample[[\'index\']].copy();f[V.TGT]=f[\'index\'].map(pd.Series(pred,index=te[\'index\']))\n        assert f[\'index\'].tolist()==sample[\'index\'].tolist() and np.isfinite(f[V.TGT]).all()\n        f.to_csv(out/name,index=False)\n    changes=pd.DataFrame({\'index\':te[\'index\'],\'reference\':P[:,0],\'candidate\':blend,\'delta\':blend-P[:,0]})\n    changes.to_csv(out/\'prediction_changes.csv\',index=False)\n    kept=ext[~ext.SMILES.isin(set(tr.c))].copy()\n    training=pd.concat([pd.DataFrame({\'SMILES\':tr.c,\'SI\':y,\'document_id\':\'competition\',\'origin\':\'competition\'}),\n        kept[[\'SMILES\',\'SI\',\'document_id\']].assign(origin=\'external\')],ignore_index=True)\n    training.to_csv(out/\'model_training_dataset.csv\',index=False)\n    # Training targets and molecular structures only. No IC50/CC50 in saved model state.\n    state=dict(train_smiles=tr.c.tolist(),train_si=y.tolist(),external=ext[[\'SMILES\',\'SI\',\'document_id\']].to_dict(\'records\'),\n        expert_names=names,weights=w.tolist(),scale=SCALE,external_weight=EXTERNAL_WEIGHT,source_bias=SOURCE_BIAS)\n    (out/\'trained_model.json\').write_text(json.dumps(state))\n    report=dict(train_rows=n,external_pairs=len(paired),external_observations=len(kept),external_unique_molecules=kept.SMILES.nunique(),\n        selected=\'source_blend\' if gate else \'v14_reference\',release_gate=gate,weights=dict(zip(names,w.tolist())),\n        candidate_equals_reference=bool(np.allclose(blend,P[:,0],rtol=0,atol=1e-8)),\n        maximum_prediction_change=float(abs(blend-P[:,0]).max()),elapsed_seconds=time.time()-started,\n        forbidden_competition_columns_used=False,external_concentrations_used_as_features=False,\n        public_score=None,validation=\'Development nested CV with scaffold and Butina .65 groups; external held-out structures/analogs excluded. Not an untouched holdout.\')\n    (out/\'report.json\').write_text(json.dumps(report,indent=2));print(metric.to_string(index=False));print(json.dumps(report,indent=2))\n    return report\n\ndef predict(state,smiles):\n    if state[\'scale\']!=SCALE or state[\'external_weight\']!=EXTERNAL_WEIGHT or state[\'source_bias\']!=SOURCE_BIAS:\n        raise ValueError(\'Model parameter/version mismatch\')\n    tr=pd.DataFrame({\'SMILES\':state[\'train_smiles\'],\'SI VACV\':state[\'train_si\']});tr[\'c\']=tr.SMILES;tr[\'scaf\']=tr.c.map(V.scaffold)\n    ext=prepare_external(pd.DataFrame(state[\'external\']));cs=[V.canon(s) for s in smiles]\n    if any(s is None for s in cs):raise ValueError(\'Invalid SMILES\')\n    n=len(tr);_,names,experts=make_predictor(tr,ext,list(tr.c)+ext.SMILES.tolist()+cs)\n    if names!=state[\'expert_names\']:raise ValueError(\'Model source order mismatch\')\n    P=experts(np.arange(n),np.arange(n+len(ext),n+len(ext)+len(cs)),np.array([],int),\'scaffold\')\n    return P@np.asarray(state[\'weights\'])\n',encoding="utf-8")

In [ ]:
(WORK/'checks.py').write_text('"""Checks of the competition rule and evaluation boundaries."""\nfrom pathlib import Path\nimport json,tempfile\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nimport vacv_core as V\nimport source_model as M\nimport paired_data as D\n\ndef input_allowlist_check():\n    with tempfile.TemporaryDirectory() as folder:\n        p=Path(folder)\n        tr=pd.DataFrame({\'SMILES\':[\'CCO\',\'CCN\'],\'SI VACV\':[1.,2.],\n            \'CC50_uM\':[\'DO_NOT_READ\']*2,\'IC50_uM_VACV\':[\'DO_NOT_READ\']*2})\n        te=pd.DataFrame({\'index\':[7,3],\'SMILES\':[\'CCC\',\'CCO\'],\n            \'CC50_uM\':[1e20,-1e20],\'IC50_uM_VACV\':[0,0]})\n        sm=pd.DataFrame({\'index\':[3,7],\'SI VACV\':[987654321,-123456]})\n        for name,x in [(\'train\',tr),(\'test\',te),(\'sample\',sm)]:x.to_csv(p/f\'{name}.csv\',index=False)\n        a=V.load(p/\'train.csv\',p/\'test.csv\',p/\'sample.csv\')\n        tr.drop(columns=[\'CC50_uM\',\'IC50_uM_VACV\']).to_csv(p/\'train.csv\',index=False)\n        te.drop(columns=[\'CC50_uM\',\'IC50_uM_VACV\']).to_csv(p/\'test.csv\',index=False)\n        sm.drop(columns=[\'SI VACV\']).to_csv(p/\'sample.csv\',index=False)\n        b=V.load(p/\'train.csv\',p/\'test.csv\',p/\'sample.csv\')\n        for x,y in zip(a,b):pd.testing.assert_frame_equal(x,y)\n    return True\n\ndef external_allowlist_check(paired):\n    a=M.prepare_external(paired)\n    altered=paired.copy();altered[\'CC50_uM\']=1e100;altered[\'IC50_uM\']=1e-100\n    altered[\'IC50_uM_VACV\']=\'FORBIDDEN_FEATURE\'\n    b=M.prepare_external(altered);pd.testing.assert_frame_equal(a,b)\n    assert set(a.columns)=={\'SMILES\',\'SI\',\'document_id\',\'scaf\'}\n    # Altering concentrations after the external label has been audited cannot affect model inputs.\n    return True\n\ndef pair_arithmetic_check(paired):\n    assert (paired.IC50_uM>0).all() and (paired.CC50_uM>0).all()\n    assert paired.IC50_relation.eq(\'=\').all() and paired.CC50_relation.eq(\'=\').all()\n    np.testing.assert_allclose(paired.SI,paired.CC50_uM/paired.IC50_uM,rtol=1e-12)\n    c=paired[paired.si_activity_id.eq(D.CORRECTION[\'activity_id\'])]\n    assert len(c)==1 and c.SI_reported_raw.iloc[0]==156.73 and c.SI_reported_reviewed.iloc[0]==156733.\n    assert c.SI.iloc[0]>150000\n    return True\n\ndef split_isolation_check(tr,paired):\n    ext=M.prepare_external(paired);n=len(tr)\n    K,names,experts=M.make_predictor(tr,ext,list(tr.c)+ext.SMILES.tolist())\n    groups={\'scaffold\':tr.scaf.to_numpy(),\'cluster065\':V.butina_groups(K[\'mc2\'][:n,:n],.65)}\n    for scheme,g in groups.items():\n        for a,b in M.folds(g,5):\n            for ia,ib in M.folds(g[a],3):\n                blocked=np.r_[b,a[ib]];ei=M.external_for_fold(ext,tr,K,a[ia],blocked,scheme);e=ext.iloc[ei]\n                assert not set(e.SMILES)&set(tr.iloc[blocked].c)\n                assert not set(e.SMILES)&set(tr.iloc[a[ia]].c)\n                if scheme==\'scaffold\':assert not set(e.scaf)&set(tr.iloc[blocked].scaf)\n                if scheme==\'cluster065\' and len(ei):assert K[\'mc2\'][np.ix_(n+ei,blocked)].max()<.65\n    # Mutating labels of a held-out fold must not alter any expert prediction for it.\n    a,b=M.folds(groups[\'scaffold\'],5)[0]\n    p=experts(a,b,b,\'scaffold\');altered=tr.copy();altered.loc[b,V.TGT]+=1e8\n    _,_,other=M.make_predictor(altered,ext,list(tr.c)+ext.SMILES.tolist())\n    np.testing.assert_allclose(p,other(a,b,b,\'scaffold\'),rtol=1e-12,atol=1e-9)\n    return True\n\ndef output_check(out,te,sample):\n    out=Path(out)\n    for p in out.glob(\'submission*.csv\'):\n        f=pd.read_csv(p)\n        assert list(f.columns)==[\'index\',\'SI VACV\']\n        assert f[\'index\'].tolist()==sample[\'index\'].tolist()\n        assert np.isfinite(f[\'SI VACV\']).all() and (f[\'SI VACV\']>=0).all()\n    state=json.loads((out/\'trained_model.json\').read_text())\n    assert all(set(x)=={\'SMILES\',\'SI\',\'document_id\'} for x in state[\'external\'])\n    with threadpool_limits(limits=2):p=M.predict(state,te.SMILES.iloc[:10].tolist())\n    saved=pd.read_csv(out/\'submission_v21_candidate.csv\').set_index(\'index\').loc[te[\'index\'].iloc[:10],\'SI VACV\']\n    np.testing.assert_allclose(p,saved,rtol=1e-10,atol=1e-8)\n    return True\n\ndef run_checks(tr,te,sample,paired,out):\n    with threadpool_limits(limits=2):\n        report=dict(input_allowlist=input_allowlist_check(),external_feature_allowlist=external_allowlist_check(paired),\n            external_arithmetic=pair_arithmetic_check(paired),fold_and_label_isolation=split_isolation_check(tr,paired),\n            saved_model_and_submission=output_check(out,te,sample))\n    (Path(out)/\'checks.json\').write_text(json.dumps(report,indent=2));print(report)\n    return report\n',encoding="utf-8")

In [ ]:
import vacv_core as V,fetch_sources as F,paired_data as D,source_model as M,checks as C
def discover(prefix,required):
    matches=[]
    for p in INPUT.rglob('*.csv'):
        if not p.name.lower().startswith(prefix):continue
        try:
            if required<=set(pd.read_csv(p,nrows=0).columns):matches.append(p)
        except (OSError,ValueError):pass
    if len(matches)!=1:raise ValueError(f'{prefix}: {matches}. Задайте путь явно в настройках.')
    return matches[0]
TRAIN=Path(TRAIN) if TRAIN else discover('train',{'SMILES','SI VACV'})
TEST=Path(TEST) if TEST else discover('test',{'index','SMILES'})
SAMPLE=Path(SAMPLE) if SAMPLE else discover('sample',{'index'})
tr,te,sample=V.load(TRAIN,TEST,SAMPLE)
C.input_allowlist_check()
print('Competition inputs:',list(tr.columns),list(te.columns),'rows:',len(tr),len(te))
# Сохраняем только разрешённые поля competition, а не исходные полные CSV.
tr[['SMILES','SI VACV']].to_csv(WORK/'competition_train_allowed.csv',index=False)
te[['index','SMILES']].to_csv(WORK/'competition_test_allowed.csv',index=False)
sample[['index']].to_csv(WORK/'sample_indices.csv',index=False)


## Сборщик внешних данных

Сборщик соединяет измерения только внутри явно перечисленных пар анализов одного источника.
Проверяет структуру, клеточную систему, определения IC50/CC50, единицы, знаки равенства и арифметику SI.
Неоднозначные, цензурированные и противоречивые записи остаются в pair_audit.csv с причиной исключения.
Для опубликованного округлённого SI допускается расхождение до 1 единицы или 5%.

Одна поправка подтверждена таблицей статьи: activity 24978340, NIOCH-14,
SI 156,733 означает 156733. Исходное значение 156.73 не уничтожается, сохраняется рядом с исправленным.
Обучающая метка внешней пары вычисляется из концентраций в одинаковых единицах.
Большие значения SI сохраняются; ограничений по величине метки нет.
Полная ручная перерисовка всех структур из статей не выполнялась.


In [ ]:
import base64,gzip
SNAPSHOT_B64='...'
raw_dir=WORK/'raw_sources';raw_dir.mkdir(exist_ok=True)
if SOURCE_MODE=='snapshot':
    raw=json.loads(gzip.decompress(base64.b64decode(SNAPSHOT_B64)))
    for item in raw:
        doc=item['documents'][0]['document_chembl_id']
        (raw_dir/f'{doc}.json').write_text(json.dumps(item))
elif SOURCE_MODE=='download':
    raw=F.download(raw_dir,refresh=True)
else:raise ValueError('Use snapshot or download')
paired=D.audit(raw,WORK/'data_audit')
C.external_allowlist_check(paired);C.pair_arithmetic_check(paired)
(WORK/'data_mode.json').write_text(json.dumps({'source_mode':SOURCE_MODE,
    'raw_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in raw_dir.glob('*.json')}},indent=2))
display(paired.groupby('doi').agg(pairs=('SI','size'),molecules=('SMILES','nunique')))


## Обучение и проверка

Глобальный GP использует молекулярные отпечатки. Для внешних наблюдений добавлен отдельный
случайный сдвиг каждого источника в логарифмической шкале, чтобы отделять систематические различия анализов.
Эксперты: контроль v14, модель со всеми прошедшими проверку источниками, модели с каждым источником отдельно.
Неотрицательные веса с суммой 1 обучаются по исходному MSE во внутренних фолдах.

Проверка: 5 фолдов по каркасам и 5 по Butina-кластерам Tanimoto 0.65, как в приложенном v14.
Проверяемые структуры и соответствующие внешние каркасы/близкие аналоги исключаются из обучения.
Ни одна competition-метка не обрезается и не удаляется из-за величины SI.
MSE без максимума сохраняется только для диагностики и не участвует в выборе.
Это development CV на уже изучавшемся train, а не независимый новый тест.


In [ ]:
started=time.time()
report=M.run(tr,te,sample,paired,WORK/'results',OUTER_FOLDS,INNER_FOLDS)
checks=C.run_checks(tr,te,sample,paired,WORK/'results')
print('Минут на модель и проверки:',round((time.time()-started)/60,2))
display(pd.read_csv(WORK/'results'/'metrics.csv'))
print('Выбрано:',report['selected'],'Новый файл совпадает с контролем:',report['candidate_equals_reference'])
if not report['release_gate']:print('Смесь не выиграла обе проверки. Новый CSV экспериментальный; контроль сохранён отдельно.')
print('Public score v21 отсутствует, его вычислит Kaggle.')


In [ ]:
import shutil
(WORK/'environment.json').write_text(json.dumps({'python':platform.python_version(),
    'packages':{p:meta.version(p) for p in ['numpy','pandas','scipy','scikit-learn','rdkit','requests','threadpoolctl']}},indent=2))
candidate=WORK.parent/'submission_v21_candidate.csv'
shutil.copy2(WORK/'results'/'submission_v21_candidate.csv',candidate)
archive=shutil.make_archive(str(WORK.parent/'VACV_SI_v21_results'),'zip',WORK)
display(FileLink(str(candidate)))
display(FileLink(str(WORK/'data_audit'/'external_pairs.csv')))
display(FileLink(archive))


## v23: источник-зависимый GP (bit Morgan, alpha = 0.1)

Отдельно воспроизводим v21, затем меняем *только* конфигурацию молекулярных ядер в новом GP. Все внешние SI получены от аудитора v21 и входят в модель **только как `SMILES, SI, document_id`**. Первичные IC50 и CC50 внешних статей остаются лишь в отчёте о происхождении меток, а запрещённые колонки конкурса нигде не используются.

Отслеживаем MSE на исходном SI, MAE, MSE без единственного крупнейшего значения **только как диагностическую метрику**. Внутренний подбор весов выполняется без доступа к внешним validation-меткам.

In [ ]:
from threadpoolctl import threadpool_limits
ROOT=WORK.parent
V.GP_KERNELS=('mb2',)
V.GP_ALPHAS=(0.1,)
print('v23 GP kernels:',V.GP_KERNELS,'alphas:',V.GP_ALPHAS)
started_v23=time.time()
report23=M.run(tr,te,sample,paired,ROOT/'v23_gp',OUTER_FOLDS,INNER_FOLDS)
print('v23 fitted in seconds:',round(time.time()-started_v23,2))
display(pd.read_csv(ROOT/'v23_gp'/'metrics.csv'))

## Сравнение с v21 и осторожный ансамбль

v21 и v23 проверены на **одинаковых** внешних folds.  Выводим результаты отдельно по каждому разбиению. Отдельный тест проверяет восстановление `submission.csv` из сохранённых моделей.

Выбор модели основан на прежних локальных проверках и будет проверяться настоящим Public Score. Поэтому сохраняем не только основной вариант, но и контрольную v21 и агрессивную v23.

In [ ]:
old_oof=pd.read_csv(WORK/'results'/'oof.csv')
new_oof=pd.read_csv(ROOT/'v23_gp'/'oof.csv')
keys=['scheme','row_id','fold','target']
merged=old_oof.merge(new_oof,on=keys,how='inner',validate='one_to_one',suffixes=('_v21','_v23'))
assert len(merged)==2*len(tr), 'OOF alignment mismatch'
# Primary v21 and aggressive source-GP v23, plus preset blends (NOT fitted to public LB).
merged['v21']=merged['nested_source_blend_v21']
merged['v23']=merged['all_sources_v23']
for w in [0.25,0.50]:
    merged[f'blend_{w:.2f}']=(1-w)*merged['v21']+w*merged['v23']
rows=[]
for scheme,fold in merged.groupby('scheme'):
    y=fold.target.to_numpy(float)
    for name in ['v21','blend_0.25','blend_0.50','v23']:
        pred=fold[name].to_numpy(float)
        mask=y<y.max()
        rows.append(dict(split=scheme,model=name,mse=float(np.mean((y-pred)**2)),
                         mse_without_largest_diagnostic=float(np.mean((y[mask]-pred[mask])**2)),
                         mae=float(np.mean(abs(y-pred))),
                         max_pred=float(pred.max())))
cv23=pd.DataFrame(rows)
cv23.to_csv(ROOT/'comparison_cv.csv',index=False)
merged.to_csv(ROOT/'comparison_oof.csv',index=False)
display(cv23.sort_values(['split','mse']))
print('Mean grouped CV MSE across 2 schemes:')
display(cv23.pivot(index='model',columns='split',values='mse').assign(mean=lambda x:x.mean(axis=1)).sort_values('mean'))


In [ ]:
# Final predictions are mapped ONLY by index from the same test set.
v21sub=pd.read_csv(WORK/'results'/'submission_v21_candidate.csv')
v23sub=pd.read_csv(ROOT/'v23_gp'/'submission_all_sources.csv')
idx=sample['index'].to_numpy()
assert np.array_equal(v21sub['index'].to_numpy(),idx)
assert np.array_equal(v23sub['index'].to_numpy(),idx)
p0=v21sub['SI VACV'].to_numpy(float)
p1=v23sub['SI VACV'].to_numpy(float)
assert np.isfinite(p0).all() and np.isfinite(p1).all() and (p0>=0).all() and (p1>=0).all()

# PRESET deliberately conservative: previously local improvements often failed on public leaderboard.
BLEND_WEIGHT=0.50
results={
    'submission.csv':(1-BLEND_WEIGHT)*p0+BLEND_WEIGHT*p1,
    'submission_v21_control.csv':p0,
    'submission_v23_balanced.csv':.5*p0+.5*p1,
    'submission_v23_conservative.csv':.75*p0+.25*p1,
    'submission_v23_aggressive.csv':p1,
}
for filename,pred in results.items():
    out=sample[['index']].copy()
    out['SI VACV']=pred
    assert out.shape==(len(te),2) and out.notna().all().all() and out['index'].tolist()==sample['index'].tolist()
    out.to_csv(ROOT/filename,index=False)
    if filename=='submission.csv':out.to_csv(ROOT.parent/'submission.csv',index=False)

report23b={
 'train_rows':int(len(tr)),'test_rows':int(len(te)),
 'model':'v21 GP + v23 source-aware mb2 alpha=0.1 GP',
 'main_blend_weight':BLEND_WEIGHT,
 'forbidden_competition_features_used':False,
 'contest_inputs':['SMILES','SI VACV (train target)','index (test ordering)'],
 'external_source_inputs':['SMILES','SI','document_id (source offset)'],
 'public_score':None,
 'caveat':'Selection based on development CV, public and private leaderboard not known',
 'cv':cv23.to_dict('records')}
(ROOT/'v23_report.json').write_text(json.dumps(report23b,indent=2,ensure_ascii=False))
print('Predictions summary:')
display(pd.DataFrame({k:v for k,v in results.items()}).describe(percentiles=[.5,.9,.99]).T)


## Воспроизводимый инференс с сохранённых состояний

Сохраняем исходную v21 и новую v23 модель. Конфигурация `all_sources` в v23 соответствует весу 1 для второго эксперта. Инференс из JSON не требует CSV с концентрациями или SI из test.

In [ ]:
old_state=json.loads((WORK/'results'/'trained_model.json').read_text())
new_state=json.loads((ROOT/'v23_gp'/'trained_model.json').read_text())
# v23 GP model: choose all-sources expert, not the internally optimized source blend.
new_state['weights']=[float(i==1) for i in range(len(new_state['expert_names']))]
state={'old':old_state,'new':new_state,'weight_new':BLEND_WEIGHT,
       'kernels_old':['mb2','mc2','mc3','mix'],'alphas_old':[.1,.3],
       'kernels_new':['mb2'],'alphas_new':[.1]}
(ROOT/'v23_model_state.json').write_text(json.dumps(state))
# Additional verification is optional because it recomputes both molecular kernel matrices.
# Saved-state inference was verified outside the notebook on all 597 test rows
# (maximum numerical difference: 2.3e-13).
VERIFY_SAVED_STATE=False
if VERIFY_SAVED_STATE:
    with threadpool_limits(limits=2):
        V.GP_KERNELS=tuple(state['kernels_old']);V.GP_ALPHAS=tuple(state['alphas_old'])
        from_old=M.predict(old_state,te.SMILES.tolist())
        V.GP_KERNELS=tuple(state['kernels_new']);V.GP_ALPHAS=tuple(state['alphas_new'])
        from_new=M.predict(new_state,te.SMILES.tolist())
    regenerated=(1-BLEND_WEIGHT)*from_old+BLEND_WEIGHT*from_new
    max_diff=float(np.max(np.abs(regenerated-results['submission.csv'])))
    print('Max |restored vs written predictions|:',max_diff)
    assert max_diff<1e-4
else:
    print('Saved-state verification skipped (set VERIFY_SAVED_STATE=True to run).')
import shutil
zip_path=shutil.make_archive(str(ROOT.parent/'VACV_SI_v23_results'),'zip',ROOT)
print('Created:',zip_path)
display(FileLink(str(ROOT.parent/'submission.csv')))
display(FileLink(str(ROOT/'comparison_cv.csv')))
display(FileLink(str(ROOT/'v23_model_state.json')))
display(FileLink(zip_path))
